# Extraction de Keywords

**TP2 – année choisie : 1960**

Nous avons choisi l'année 1960 pour trois raisons. D'abord, c'est une année très riche pour la Belgique : le Congo devient indépendant le 30 juin, puis une crise éclate. Ensuite, la fin de l'année est marquée par le projet de « loi unique » et par le mariage du roi Baudouin. Enfin, le corpus contient 100 fichiers pour 1960 et ils couvrent les 12 mois. L'année est donc bien représentée.

Par rapport au notebook du cours, nous avons changé deux choses :
- le chemin vers les données, car le notebook se trouve maintenant dans `tps/tp2` ;
- la liste des fichiers, car nous gardons seulement ceux de 1960.

## Imports

In [1]:
import os
from collections import Counter
import yake

## Extraire les mots clés d'un document avec Yake

https://github.com/LIAAD/yake

In [2]:
# Instantier l'extracteur de mots clés
kw_extractor = yake.KeywordExtractor(lan="fr", top=50)
kw_extractor

In [3]:
# Choisir une année
year = 1960

In [4]:
# Lister les fichiers de l'année choisie
# Le motif "_1960-" correspond à la date dans le nom du fichier (ex: KB_JB838_1960-01-10_01-00018.txt)
data_path = "../../data/txt/"
files = sorted([f for f in os.listdir(data_path) if f.endswith('.txt') and f"_{year}-" in f])

Nous cherchons le motif `_1960-` et pas seulement `1960`. En effet, ce motif correspond à la date dans le nom du fichier. Nous évitons ainsi de prendre un fichier d'une autre année qui contiendrait « 1960 » ailleurs dans son nom.

In [5]:
# Imprimer le nombre de fichiers identifiés
len(files)

100

In [6]:
# Les dix premiers fichiers
files[:10]

['KB_JB838_1960-01-10_01-00018.txt',
 'KB_JB838_1960-01-12_01-00011.txt',
 'KB_JB838_1960-01-13_01-00009.txt',
 'KB_JB838_1960-01-21_01-00014.txt',
 'KB_JB838_1960-01-22_01-00015.txt',
 'KB_JB838_1960-01-27_01-00006.txt',
 'KB_JB838_1960-01-28_01-00011.txt',
 'KB_JB838_1960-01-28_01-00021.txt',
 'KB_JB838_1960-01-30_01-00007.txt',
 'KB_JB838_1960-01-31_01-00024.txt']

Nous choisissons la première page du journal du 10 juillet 1960. Cette date tombe dix jours après l'indépendance du Congo, donc nous nous attendons à y trouver des mots clés liés à cet événement.

In [7]:
# Choisir un fichier
this_file = "KB_JB838_1960-07-10_01-00001.txt"
this_file in files

True

In [8]:
# Récupérer le texte du fichier
text = open(os.path.join(data_path, this_file), 'r', encoding='utf-8').read()
text[:500]

'74* ANNEE — N" 165 t • • JOURNAL QUOTIDIEN 24 Pages - 2 fr. Congo 4 It. Tronc* r N. F. 0 h. 30 — Sults* l 0 Ir. 40 triste» , CHEQUES POSTAUX PUBLICITE t Compte n* 567S ftOSSEL & C 1 * VENTE I Soit et Soir llluilri ! 14686 Rossel JC" PONS aux Œuvre» du Soit t 14232 Rossel & C" les manuscrits non Insérés ne sont pas rendus LE t AGENCE ROSSEL - Annonces et Publicité LE SOIR - Vente et Abonnements i BRUXELLES I t* 112, rue Royale. 112 PARIS (8\'t t 73. r. d\'Anjou. T. EURope 36.16 PLACE DE LOUVAIN, 21'

In [9]:
# Extraire les mots clés de ce texte
keywords = kw_extractor.extract_keywords(text)
keywords

[('Congo', np.float64(0.011459400532906791)),
 ('belges', np.float64(0.017069287223451187)),
 ('loi belge', np.float64(0.020033090513464708)),
 ('belge', np.float64(0.020862462161995894)),
 ('Force publique', np.float64(0.021580863157646284)),
 ('FORCES BELGES', np.float64(0.02226620235853606)),
 ('troupes belges', np.float64(0.022746122969216475)),
 ('Elisabethville', np.float64(0.02479200991256395)),
 ('gouvernement', np.float64(0.02504215288323947)),
 ('FORCES BELGES IHIIRVIEMHEHT', np.float64(0.02648155097544798)),
 ('qu’il', np.float64(0.029938688471765413)),
 ('Congo belge', np.float64(0.03105856165079044)),
 ('JOURNAL QUOTIDIEN', np.float64(0.032037844002974915)),
 ('forces belges aurait', np.float64(0.03250291290386069)),
 ('loi', np.float64(0.03354842151402469)),
 ('Katanga', np.float64(0.03464835253062551)),
 ('BRUXELLES', np.float64(0.03943590688125622)),
 ('soldats', np.float64(0.039737260840907784)),
 ('troupes', np.float64(0.04252677075634702)),
 ('DIMANCHE', np.float64(0

In [10]:
# Ne garder que les bigrammes
kept = []
for kw, score in keywords:
    words = kw.split()
    if len(words) == 2:
        kept.append(kw)
kept

['loi belge',
 'Force publique',
 'FORCES BELGES',
 'troupes belges',
 'Congo belge',
 'JOURNAL QUOTIDIEN',
 'Société libre',
 'soldats belges',
 'qu’il faut']

**Commentaire.** Les bigrammes de cette page parlent presque tous du même sujet : « Force publique », « troupes belges », « soldats belges », « Congo belge ». La page raconte donc l'envoi de soldats belges au Congo après la révolte de l'armée congolaise. Yake résume bien le thème de la page. En revanche, il garde aussi des expressions sans intérêt comme « journal quotidien », car elles sont imprimées dans l'en-tête du journal.

## Faire la même opération sur tous les documents de l'année

In [11]:
# On garde les bigrammes de chaque fichier dans un dictionnaire pour pouvoir les compter ensuite
keywords_by_file = {}
for f in files:
    text = open(os.path.join(data_path, f), 'r', encoding="utf-8").read()
    keywords = kw_extractor.extract_keywords(text)
    kept = []
    for kw, score in keywords:
        words = kw.split()
        if len(words) == 2:
            kept.append(kw)
    keywords_by_file[f] = kept
    print(f"{f} mentions these keywords: {', '.join(kept)}...")

KB_JB838_1960-01-10_01-00018.txt mentions these keywords: Agence Rossel, l’Agence Rossel, Ecrire Agence, Faire offre, Rossel num, IMPORTANTE SOCIETE, Importante firme, BRUXELLES CHERCHE, Importante société, Firme Importante, curriculum vitae, offre manuscrite, vltae Agence...


KB_JB838_1960-01-12_01-00011.txt mentions these keywords: Van den, Van der, classements officiels, Coq Anglais, Van Houtte, Country Club, Léopold Club, Van Blesebroeck, Astrld Club, Van Belle, Baudouin Club...


KB_JB838_1960-01-13_01-00009.txt mentions these keywords: VAN LOOY, Van Stcenbergen, Rik Van, DIVISION III, Fort Jaco, Grand Prix, grand temps, Dimanche division, dimanche dernier...


KB_JB838_1960-01-21_01-00014.txt mentions these keywords: chemise lavée, CHEMISES LAVÉES, Office Propriétaires, Propriétaires Tél, WATERLOO TéL, maison rapport, belle maison, PRIX TARIF, francs Office, bonne maison...


KB_JB838_1960-01-22_01-00015.txt mentions these keywords: LOUER App, Pastilles Antibiotiques, FER FORGE, spac. app, Pet. app...


KB_JB838_1960-01-27_01-00006.txt mentions these keywords: Commerce extérieur, Haute Autorité, Marché commun, politique scientifique, Commission scientifique, recherche scientifique, Conseil national...


... (87 outputs left out)

KB_JB838_1960-12-06_01-00011.txt mentions these keywords: Grand Prix, derby Anderlecht, Van Kerrebroek, willy Mairesse, Mairesse fit, Van Geel...


KB_JB838_1960-12-10_01-00019.txt mentions these keywords: Biscuits Rem, Biscuiterie Alsacienne, Alsacienne Confiserie, Charcuterie Alsacienne, Alsacienne Fincker, Artus Biscuits, Amieux Rivoir, Carret Chocolat, Chocolat Cemoi, Cemoi Biscottes, Biscottes Darry, Darry Alsaglace, Alsaglace Feyel, Feyel Cassegrain, Rem Biscuiterie, Jaumart Charcuterie, Marquise Amora, Battendî.er Mîlliat, Mîlliat Frères...


KB_JB838_1960-12-21_01-00017.txt mentions these keywords: Ets BRACONNIER, BRACONNIER Siège, Agence Rossel, Rossel num, Ecrire Agence, Siège central, DACTYLO BILINGUE, Agença Rossel, Bruxelles dem, STENO DACTYLO...


KB_JB838_1960-12-23_01-00019.txt mentions these keywords: l'Agence Rossel, som Moment, Moment critique, Rossel num, Agence Rossel, Importante firme, service militaire, entrée immédiate, Firme dem, demande VENDEUR, critique dons, place demande, IMPORTANTE MAISON...


## Compter les mots clés qui reviennent dans plusieurs documents

La liste ci-dessus est longue, donc elle est difficile à lire. Nous comptons alors dans combien de fichiers chaque bigramme apparaît. Cela montre les sujets qui reviennent le plus souvent pendant l'année.

In [12]:
# Pour chaque bigramme, compter dans combien de fichiers de l'année il apparaît
counter = Counter()
for f, kws in keywords_by_file.items():
    counter.update(set(kw.lower() for kw in kws))

for kw, n in counter.most_common(40):
    print(f"{kw} : {n} fichiers")

agence rossel : 17 fichiers
ecrire agence : 11 fichiers
l’agence rossel : 9 fichiers
rossel num : 9 fichiers
dem. fem : 7 fichiers
faire offre : 6 fichiers
van den : 6 fichiers
van looy : 6 fichiers
congo belge : 6 fichiers
new york : 6 fichiers
bruxelles tél : 6 fichiers
importante firme : 5 fichiers
demeurant rue : 5 fichiers
bruxelles demande : 5 fichiers
curriculum vitae : 4 fichiers
rik van : 4 fichiers
grand prix : 4 fichiers
opel rek : 4 fichiers
journal quotidien : 4 fichiers
habitant rue : 4 fichiers
bruxelles dem : 4 fichiers
bel app : 4 fichiers
nations unies : 4 fichiers
offre manuscrite : 3 fichiers
division iii : 3 fichiers
propriétaires tél : 3 fichiers
bonne maison : 3 fichiers
marché commun : 3 fichiers
rue van : 3 fichiers
soir mercredi : 3 fichiers
conseil communal : 3 fichiers
d’une fracture : 3 fichiers
bon salaire : 3 fichiers
jeune homme : 3 fichiers
domicilié rue : 3 fichiers
reine elisabeth : 3 fichiers
lundi matin : 3 fichiers
servante int : 3 fichiers
gouvern

**Commentaire.** Trois groupes de mots clés ressortent.

1. **Les petites annonces.** « Agence Rossel » apparaît dans 17 fichiers sur 100, donc dans près d'une page sur cinq. « Ecrire agence », « faire offre » et « curriculum vitae » vont dans le même sens. Ces mots viennent des pages d'offres d'emploi. Ils sont fréquents, mais ils ne disent rien sur l'actualité de 1960. Nous devrons donc les retirer pour le nuage de mots.
2. **L'actualité politique.** « Congo belge » et « New York » apparaissent chacun dans 6 fichiers, « Nations Unies » dans 4, « gouvernement belge » et « marché commun » dans 3. Le Congo est donc le sujet d'actualité le plus présent. De plus, « New York » et « Nations Unies » montrent que la crise congolaise se discute aussi à l'ONU.
3. **Le sport.** « Van Looy » apparaît dans 6 fichiers et « Rik Van » dans 4. Il s'agit du coureur cycliste Rik Van Looy, qui est alors une vedette en Belgique. « Grand prix » (4 fichiers) confirme la place du sport dans le journal.

**Limites.** Certains bigrammes n'ont pas de sens, par exemple « van den » ou « rue van ». Ce sont des morceaux de noms flamands. D'autres sont déformés par l'OCR, par exemple « vltae agence ». Les résultats de Yake doivent donc être relus avant d'être interprétés.